In [10]:
import pandas as pd
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
import sqlite3

print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)

Pandas version: 2.0.3
Numpy version: 1.24.4


In [11]:
df = pd.read_csv('data/dataset_clean1.csv')
print(df.shape)
df.head()

(10000, 13)


,event_id,username,role,status,timestamp,ip_address,location,device,account_status,login_attempts,source_port,last_success_login,alert_flag
0,1,root,user,failed,2024-08-06 16:50:00,165.93.148.165,Germany,Unknown,inactive,Unknown,443,2024-11-06 16:50:00,0
1,2,jdoe,guest,failed,2024-06-19 22:06:00,221.49.14.175,Spain,Ubuntu,locked,5,443,2024-06-06 22:06:00,1
2,3,service,user,failed,2024-04-08 22:14:00,242.73.33.181,Spain,Windows,active,3,3389,2024-12-06 22:14:00,0
3,4,admin,guest,failed,2024-06-18 07:47:00,251.233.216.87,Spain,Windows,active,4,unkown,2024-05-20 07:47:00,0
4,5,backup,admin,failed,2024-04-22 10:04:00,205.59.238.115,Spain,Windows,active,5,8080,2024-02-29 10:04:00,0


In [17]:
# Creación de las 3 dimensiones , usuarios,puertos,fechas
#  usuarios
dim_users = df[['username', 'role', 'location', 'device']].drop_duplicates().reset_index(drop=True)
dim_users['user_id'] = dim_users.index + 1  # ID secuencial


In [18]:

# Dimensión de puertos
dim_ports = df[['source_port']].drop_duplicates().reset_index(drop=True)
dim_ports['port_id'] = dim_ports.index + 1


In [19]:

# Dimensión de fechas
dim_dates = df[['timestamp']].drop_duplicates().reset_index(drop=True)
dim_dates['timestamp'] = pd.to_datetime(dim_dates['timestamp'])
dim_dates['date_id'] = dim_dates.index + 1
dim_dates['year'] = dim_dates['timestamp'].dt.year
dim_dates['month'] = dim_dates['timestamp'].dt.month
dim_dates['day'] = dim_dates['timestamp'].dt.day


In [ ]:
# Crear la tabla de hechos : 
fact_logins = df.merge(dim_users, on=['username', 'role', 'location', 'device'], how='left') \
                .merge(dim_ports, on='source_port', how='left') \
                .merge(dim_dates, on='timestamp', how='left')

# Seleccionar columnas finales de la tabla de hechos
fact_logins = fact_logins[['user_id', 'port_id', 'date_id', 'status', 'login_attempts', 'alert_flag']]


In [ ]:
# Añadir ID de hecho
fact_logins['fact_id'] = fact_logins.index + 1


# Crear conexión
engine = create_engine('sqlite:///warehouse_pandas.db')

# Guardar dimensiones y hechos
dim_users.to_sql('dim_users', engine, index=False, if_exists='replace')
dim_ports.to_sql('dim_ports', engine, index=False, if_exists='replace')
dim_dates.to_sql('dim_dates', engine, index=False, if_exists='replace')
fact_logins.to_sql('fact_logins', engine, index=False, if_exists='replace')

print("Tablas guardadas correctamente en warehouse_pandas.db")


In [12]:
# Conectar a la base de datos
conn = sqlite3.connect("warehouse/warehouse_pandas.db")
cursor = conn.cursor()
print("\nDim Users:")
print(pd.read_sql_query("SELECT * FROM dim_users LIMIT 5;", conn))





Dim Users:
  username   role location   device  user_id
0     root   user  Germany  Unknown        1
1     jdoe  guest    Spain   Ubuntu        2
2  service   user    Spain  Windows        3
3    admin  guest    Spain  Windows        4
4   backup  admin    Spain  Windows        5


In [13]:

print("\nDim Dates:")
print(pd.read_sql_query("SELECT * FROM dim_dates LIMIT 5;", conn))



Dim Dates:
                    timestamp  date_id  year  month  day
0  2024-08-06 16:50:00.000000        1  2024      8    6
1  2024-06-19 22:06:00.000000        2  2024      6   19
2  2024-04-08 22:14:00.000000        3  2024      4    8
3  2024-06-18 07:47:00.000000        4  2024      6   18
4  2024-04-22 10:04:00.000000        5  2024      4   22


In [6]:

print("\nFact Logins:")
print(pd.read_sql_query("SELECT * FROM fact_logins LIMIT 5;", conn))



Fact Logins:
   user_id  port_id  date_id  status login_attempts  alert_flag  fact_id
0        1        1        1  failed        Unknown           0        1
1        2        1        2  failed              5           1        2
2        3        2        3  failed              3           0        3
3        4        3        4  failed              4           0        4
4        5        4        5  failed              5           0        5


In [5]:
conn.close()
